# PA2 Notebook 1: Training

Run on Kaggle with GPU T4 x2 and internet on (Save Version -> Save & Run All).

Sets up the repo, prepares the data, and trains all four models one after another: SFT on Qwen2.5-0.5B,
DPO on HH-RLHF, DPO on UltraFeedback, and the LoRA persona model with TRL. Everything is saved under
`outputs/` so the next notebook can use it.

## Setup

In [ ]:
# clone the repo (needs internet on)
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """run a shell command, print its output, and stop if it fails"""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode} (see output above): {cmd}")

In [ ]:
# install packages without reinstalling torch
# peft/trl are for the LoRA part; torchao is removed because it clashes with peft
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11.0" "trl>=0.24,<0.30"
!python -m pip install -q -e . --no-deps

In [ ]:
# check both T4s are there and print versions
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
# run the adapter tests
sh("PYTHONPATH=. python -m pytest -q tests/")

## Data preparation

In [ ]:
# build the SFT, HH and UltraFeedback subsets (seed 42)
# also saves the chosen ids and filter counts in data/manifests/
sh("python scripts/prepare_data.py --out_dir data/processed --which sft hh uf")
!wc -l data/processed/*.jsonl
!cat data/manifests/*_stats.json

In [ ]:
# quick test with a tiny model before the real runs
sh("python scripts/smoke_train.py --work_dir /tmp/pa2_smoke")

## SFT

In [ ]:
sh("python scripts/train_sft.py --config configs/sft.yaml")

In [ ]:
!cat outputs/sft/summary.json
sh("python scripts/plot_metrics.py --run sft=outputs/sft --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/sft_loss.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/sft_run_outputs.zip sft -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/sft_run_outputs.zip && du -sh outputs/sft

## DPO on HH-RLHF

Policy and reference both start from the SFT best checkpoint (policy on cuda:0, frozen reference in fp16 on
cuda:1). beta = 0.1, lr = 1e-6, batch size 1 with 32 accumulation steps.

In [ ]:
sh("python scripts/train_dpo.py --config configs/dpo_hh.yaml --set sft_checkpoint=outputs/sft/best")

In [ ]:
!cat outputs/dpo_hh/summary.json
sh("python scripts/plot_metrics.py --run dpo_hh=outputs/dpo_hh --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/dpo_hh_metrics.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/dpo_hh_run_outputs.zip dpo_hh -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/dpo_hh_run_outputs.zip && du -sh outputs/dpo_hh

## DPO on UltraFeedback

Same code and settings as the HH run.

In [ ]:
sh("python scripts/train_dpo.py --config configs/dpo_uf.yaml --set sft_checkpoint=outputs/sft/best")

In [ ]:
!cat outputs/dpo_uf/summary.json
sh("python scripts/plot_metrics.py --run dpo_uf=outputs/dpo_uf --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/dpo_uf_metrics.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/dpo_uf_run_outputs.zip dpo_uf -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/dpo_uf_run_outputs.zip && du -sh outputs/dpo_uf

## LoRA persona DPO (TRL)

SmolLM2-360M-Instruct trained on `ramsay_style_preferences.json`, with 22 pairs held out for validation.

In [ ]:
sh("python scripts/train_lora_dpo.py --config configs/lora_dpo.yaml")

In [ ]:
!cat outputs/lora_dpo/summary.json
sh("python scripts/plot_metrics.py --run lora_dpo=outputs/lora_dpo --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/lora_dpo_metrics.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/lora_dpo_run_outputs.zip lora_dpo -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/lora_dpo_run_outputs.zip && du -sh outputs/lora_dpo

## Plots

In [ ]:
sh("python scripts/plot_metrics.py --run sft=outputs/sft --run dpo_hh=outputs/dpo_hh "
   "--run dpo_uf=outputs/dpo_uf --run lora_dpo=outputs/lora_dpo --out_dir outputs/plots")
!cd outputs && zip -q -r /kaggle/working/plots.zip plots
from IPython.display import Image, display
display(Image("outputs/plots/dpo_runs_comparison.png"))

Done. The output of this notebook is used as the input for notebook 2.